In [ ]:
# ¿Las probabilidades estimadas corresponden a tasas de default observadas?

import pandas as pd

predictions = pd.read_csv("../data/risk_predictions.csv")
predictions["probability_band"] = pd.cut(predictions["default_probability"], bins=[0, .2, .4, .6, .8, 1], include_lowest=True)
calibration = predictions.groupby("probability_band", observed=True).agg(applications=("application_id", "size"), mean_probability=("default_probability", "mean"), observed_default_rate=("observed_default", "mean")).reset_index()
calibration

In [ ]:
# ¿Qué umbral equilibra el costo de no priorizar un default y el costo de una revisión innecesaria?

cost_false_negative = 10
cost_false_positive = 1
rows = []
for threshold in [.2, .3, .4, .5, .6]:
    priority = predictions.default_probability >= threshold
    false_negative = ((~priority) & (predictions.observed_default == 1)).sum()
    false_positive = (priority & (predictions.observed_default == 0)).sum()
    rows.append({"threshold": threshold, "prioritized_applications": priority.sum(), "false_negative": false_negative, "false_positive": false_positive, "expected_cost": cost_false_negative * false_negative + cost_false_positive * false_positive})
tradeoff = pd.DataFrame(rows)
tradeoff

In [ ]:
# ¿La política elegida produce diferencias que deben revisarse entre grupos?

selected_threshold = tradeoff.loc[tradeoff.expected_cost.idxmin(), "threshold"]
predictions["prioritized"] = predictions.default_probability >= selected_threshold
group_review = predictions.groupby("sex").agg(applications=("application_id", "size"), observed_default_rate=("observed_default", "mean"), prioritization_rate=("prioritized", "mean"), mean_probability=("default_probability", "mean")).reset_index()
group_review

In [ ]:
# Se conservan los elementos necesarios para auditar la política de priorización.

from pathlib import Path

output = Path("../submission")
calibration.to_csv(output / "calibration_summary.csv", index=False)
tradeoff.to_csv(output / "threshold_tradeoff.csv", index=False)
group_review.to_csv(output / "group_review.csv", index=False)